In [ ]:
# ============================================================
# TALTECH AI PIPELINE
# NOTEBOOK 06 — COMMON SUPERVISED DATASET + CHRONOLOGICAL SPLIT
# Portable Windows/Linux
#
# Purpose:
# 1) Start from the cleaned complete-case synchronized dataset.
# 2) Create the common target: temperature at t+15 min.
# 3) Never create a target across a temporal gap/segment boundary.
# 4) Apply ONE global chronological Train/Validation/Test split.
# 5) Perform NO scaling here (to avoid leakage).
#
# INPUT:
# outputs/05_ml_dataset/06_ai_ready_complete_case.csv
#
# OUTPUT:
# outputs/06_common_experiment/
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
INPUT_FILE = (
    OUTPUT_DIR
    / "05_ml_dataset"
    / "06_ai_ready_complete_case.csv"
)
OUT = OUTPUT_DIR / "06_common_experiment"
OUT.mkdir(parents=True, exist_ok=True)

print("====================================")
print("NOTEBOOK 06 — COMMON EXPERIMENT")
print("====================================")
print("Project :", PROJECT_DIR)
print("Input   :", INPUT_FILE)
print("Output  :", OUT)

assert INPUT_FILE.exists(), (
    f"\nInput file not found:\n{INPUT_FILE}\n"
    "Run Notebook 05 first."
)

# ============================================================
# 2. FIXED EXPERIMENTAL PARAMETERS
# ============================================================

STEP = pd.Timedelta(minutes=15)
HORIZON_MINUTES = 15

# Chronological proportions.
# Boundaries are determined from UNIQUE TIMESTAMPS, not rows,
# so all devices at a given timestamp belong to the same split.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# ============================================================
# 3. LOAD COMPLETE-CASE DATA
# ============================================================

df = pd.read_csv(INPUT_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True,
    errors="coerce"
)

for c in ["temperature", "relative_humidity", "co2"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

required = {
    "uuid", "timestamp",
    "temperature", "relative_humidity", "co2",
    "segment_id"
}

missing_cols = required - set(df.columns)
assert not missing_cols, f"Missing columns: {sorted(missing_cols)}"

assert df["timestamp"].notna().all()
assert df[["temperature", "relative_humidity", "co2"]].notna().all().all()

df = df.sort_values(
    ["uuid", "segment_id", "timestamp"]
).reset_index(drop=True)

print("\nRows loaded :", len(df))
print("Devices     :", df["uuid"].nunique())
print("Segments    :", df[["uuid", "segment_id"]].drop_duplicates().shape[0])
print("Period      :", df["timestamp"].min(), "->", df["timestamp"].max())

# ============================================================
# 4. CREATE COMMON t+15 MIN TARGET
# ============================================================
# The target is valid only if:
# - next row belongs to same UUID
# - next row belongs to same contiguous segment
# - next timestamp is EXACTLY t + 15 min
#
# This prevents forecasting samples from crossing long gaps.

g = df.groupby(
    ["uuid", "segment_id"],
    sort=False,
    group_keys=False
)

df["target_timestamp"] = g["timestamp"].shift(-1)
df["target_temperature_t_plus_15"] = g["temperature"].shift(-1)

df["target_delta_minutes"] = (
    (df["target_timestamp"] - df["timestamp"])
    .dt.total_seconds()
    / 60
)

valid_target = (
    df["target_timestamp"].notna()
    &
    (df["target_timestamp"] == df["timestamp"] + STEP)
    &
    df["target_temperature_t_plus_15"].notna()
)

supervised = df.loc[valid_target].copy()

# Target provenance columns make the prediction horizon explicit.
supervised["forecast_horizon_minutes"] = HORIZON_MINUTES

print("\n====================================")
print("COMMON TARGET")
print("====================================")
print("Complete-case rows before target :", len(df))
print("Valid t+15 supervised samples    :", len(supervised))
print("Rows excluded at segment ends    :", len(df) - len(supervised))
print(
    "Target delta unique values       :",
    sorted(supervised["target_delta_minutes"].unique())
)

assert (
    supervised["target_delta_minutes"] == HORIZON_MINUTES
).all()

# ============================================================
# 5. GLOBAL CHRONOLOGICAL SPLIT BOUNDARIES
# ============================================================
# IMPORTANT:
# We split by unique input timestamps.
# No random split is used.
# The same timestamp can never appear in different partitions.

unique_times = np.array(
    sorted(supervised["timestamp"].unique())
)

n_times = len(unique_times)

assert n_times >= 3, "Not enough timestamps for temporal splitting."

train_end_idx = int(np.floor(n_times * TRAIN_RATIO)) - 1
val_end_idx = int(np.floor(n_times * (TRAIN_RATIO + VAL_RATIO))) - 1

train_end_idx = max(0, train_end_idx)
val_end_idx = max(train_end_idx + 1, val_end_idx)
val_end_idx = min(val_end_idx, n_times - 2)

train_end = pd.Timestamp(unique_times[train_end_idx])
val_end = pd.Timestamp(unique_times[val_end_idx])

supervised["split"] = np.select(
    [
        supervised["timestamp"] <= train_end,
        (
            (supervised["timestamp"] > train_end)
            &
            (supervised["timestamp"] <= val_end)
        )
    ],
    [
        "train",
        "validation"
    ],
    default="test"
)

# ============================================================
# 6. LEAKAGE CHECKS
# ============================================================

train = supervised.loc[supervised["split"] == "train"].copy()
val = supervised.loc[supervised["split"] == "validation"].copy()
test = supervised.loc[supervised["split"] == "test"].copy()

assert len(train) > 0 and len(val) > 0 and len(test) > 0

assert train["timestamp"].max() < val["timestamp"].min()
assert val["timestamp"].max() < test["timestamp"].min()

# Stronger forecasting leakage rule:
# An input belongs to a split only if its TARGET timestamp also
# stays within the same temporal partition.
#
# This removes boundary samples whose y lies in the next split.

def same_partition_target_mask(frame, split_name):
    if split_name == "train":
        return frame["target_timestamp"] <= train_end

    if split_name == "validation":
        return (
            (frame["target_timestamp"] > train_end)
            &
            (frame["target_timestamp"] <= val_end)
        )

    return frame["target_timestamp"] > val_end

before_boundary_guard = len(supervised)

keep = pd.Series(False, index=supervised.index)

for split_name in ["train", "validation", "test"]:
    idx = supervised.index[supervised["split"] == split_name]
    keep.loc[idx] = same_partition_target_mask(
        supervised.loc[idx],
        split_name
    )

supervised = supervised.loc[keep].copy()

train = supervised.loc[supervised["split"] == "train"].copy()
val = supervised.loc[supervised["split"] == "validation"].copy()
test = supervised.loc[supervised["split"] == "test"].copy()

boundary_removed = before_boundary_guard - len(supervised)

# ============================================================
# 7. SPLIT AUDIT
# ============================================================

audit_rows = []

for split_name, part in [
    ("train", train),
    ("validation", val),
    ("test", test)
]:
    audit_rows.append({
        "split": split_name,
        "samples": len(part),
        "sample_pct": 100 * len(part) / len(supervised),
        "devices": part["uuid"].nunique(),
        "segments_represented":
            part[["uuid", "segment_id"]].drop_duplicates().shape[0],
        "input_start": part["timestamp"].min(),
        "input_end": part["timestamp"].max(),
        "target_start": part["target_timestamp"].min(),
        "target_end": part["target_timestamp"].max(),
        "temperature_mean": part["temperature"].mean(),
        "target_temperature_mean":
            part["target_temperature_t_plus_15"].mean(),
        "target_temperature_std":
            part["target_temperature_t_plus_15"].std()
    })

audit = pd.DataFrame(audit_rows)

print("\n====================================")
print("CHRONOLOGICAL SPLIT")
print("====================================")
print("Unique supervised timestamps :", n_times)
print("Train input cutoff           :", train_end)
print("Validation input cutoff      :", val_end)
print("Boundary samples removed     :", boundary_removed)
print()
print(audit.to_string(index=False))

# ============================================================
# 8. DEVICE-LEVEL SPLIT COUNTS
# ============================================================

device_counts = (
    supervised
    .groupby(["split", "uuid"])
    .size()
    .rename("samples")
    .reset_index()
)

device_counts.to_csv(
    OUT / "02_samples_by_split_device.csv",
    index=False
)

# ============================================================
# 9. SAVE COMMON DATASETS
# ============================================================

common_columns = [
    "uuid",
    "timestamp",
    "segment_id",
    "temperature",
    "relative_humidity",
    "co2",
    "target_timestamp",
    "target_temperature_t_plus_15",
    "forecast_horizon_minutes",
    "split"
]

supervised = supervised[
    common_columns
].sort_values(
    ["timestamp", "uuid"]
).reset_index(drop=True)

supervised.to_csv(
    OUT / "01_common_supervised_dataset.csv",
    index=False
)

for split_name in ["train", "validation", "test"]:
    supervised.loc[
        supervised["split"] == split_name
    ].to_csv(
        OUT / f"03_{split_name}_common.csv",
        index=False
    )

audit.to_csv(
    OUT / "04_chronological_split_audit.csv",
    index=False
)

# ============================================================
# 10. PROTOCOL MANIFEST
# ============================================================

manifest = pd.DataFrame([
    {
        "parameter": "prediction_target",
        "value": "indoor temperature at t+15 min"
    },
    {
        "parameter": "forecast_horizon_minutes",
        "value": 15
    },
    {
        "parameter": "input_variables_at_t",
        "value": "temperature, relative_humidity, co2"
    },
    {
        "parameter": "target_validity",
        "value":
            "same UUID + same contiguous segment + exact 15-min next timestamp"
    },
    {
        "parameter": "split_strategy",
        "value":
            "global chronological split using unique input timestamps"
    },
    {
        "parameter": "nominal_split_ratio",
        "value": "70% train / 15% validation / 15% test"
    },
    {
        "parameter": "train_end",
        "value": str(train_end)
    },
    {
        "parameter": "validation_end",
        "value": str(val_end)
    },
    {
        "parameter": "boundary_leakage_guard",
        "value":
            "target timestamp must remain inside same partition"
    },
    {
        "parameter": "scaling",
        "value":
            "not performed; scaler must be fitted on training data only"
    },
    {
        "parameter": "random_shuffle",
        "value": "disabled"
    },
    {
        "parameter": "common_protocol",
        "value":
            "same target and chronological partitions for all representation-model pipelines"
    }
])

manifest.to_csv(
    OUT / "05_experimental_protocol_manifest.csv",
    index=False
)

# ============================================================
# 11. MACHINE-READABLE SPLIT METADATA
# ============================================================

metadata = {
    "source_file": str(INPUT_FILE),
    "target": "temperature_t_plus_15_min",
    "forecast_horizon_minutes": 15,
    "train_ratio_nominal": TRAIN_RATIO,
    "validation_ratio_nominal": VAL_RATIO,
    "test_ratio_nominal": TEST_RATIO,
    "train_end": str(train_end),
    "validation_end": str(val_end),
    "samples_total": int(len(supervised)),
    "samples_train": int((supervised["split"] == "train").sum()),
    "samples_validation": int((supervised["split"] == "validation").sum()),
    "samples_test": int((supervised["split"] == "test").sum()),
    "boundary_samples_removed": int(boundary_removed),
    "scaling_applied": False
}

with (
    OUT / "06_split_metadata.json"
).open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ============================================================
# 12. FINAL CHECKS
# ============================================================

print("\n====================================")
print("FINAL VALIDATION")
print("====================================")

assert supervised[
    ["temperature", "relative_humidity", "co2",
     "target_temperature_t_plus_15"]
].notna().all().all()

assert (
    pd.to_datetime(supervised["target_timestamp"], utc=True)
    -
    pd.to_datetime(supervised["timestamp"], utc=True)
    ==
    STEP
).all()

assert supervised.duplicated(
    subset=["uuid", "timestamp"]
).sum() == 0

print("✓ No missing predictors in common supervised samples")
print("✓ Every target is exactly t+15 min")
print("✓ No target crosses a temporal gap")
print("✓ No target crosses Train/Validation/Test boundaries")
print("✓ Global chronological split fixed")
print("✓ No random shuffle")
print("✓ No scaling performed")

print("\n====================================")
print("FILES CREATED")
print("====================================")
for f in sorted(OUT.iterdir()):
    print(" -", f.name)
